# 02 · Data Preprocessing: Road Traffic Accident (RTA) Dataset

**Project:** An Intelligent Traffic Violation Management System for Sri Lanka with ML-Based Dynamic Severity Assessment and Driver Risk Scoring
**Author:** Damma Monnankulama (20211211 / W1899305)

This notebook turns the raw dataset into clean, model-ready data (Proposal §3.4.2: *Outputs: a cleaned feature set (X) and target variable (y)*). Every step comes from a finding in `01_data_exploration.ipynb`.

| Step | What | Why |
|---|---|---|
| 1 | Normalise text | Stray whitespace splits one category into several |
| 2 | Fix corrupted / inconsistent labels | Data-entry damage (`?`, `statioNot a Pedestrianry`, merged labels) |
| 3 | Unify missing values → `Unknown` | NaN, `na`, `unknown` all mean the same thing |
| 4 | Feature engineering from `Time` / `Day_of_week` | The fatal rate depends strongly on time of day |
| 5 | Encode the target ordinally | Slight < Serious < Fatal |
| 6 | Feature selection for Model 1 | Remove post-accident leakage and sensitive or sparse columns |
| 7 | Validation checks | Prove the cleaning is correct and lossless |
| 8 | Save to `data/processed/` + metadata | Reproducibility (NFR-4, §3.4.5) |

> **Deliberately *not* done here:** one-hot encoding, scaling, train/test split and SMOTE.
> They must be *fitted on the training split only*, otherwise test-set information leaks into training. They belong in the modelling notebook (`03_...`) inside an sklearn `Pipeline`.

## 0. Setup

In [1]:
# --- Environment & paths -------------------------------------------------
# Works both locally (conda env `accident-severity-poc`) and on Google Colab.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: upload the project folder to Google Drive, then set its path here.
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fyp-poc-v4")
else:
    # Locally: notebooks live in <project>/notebooks
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_PATH       = PROJECT_ROOT / "data" / "raw" / "RTA Dataset.csv"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR    = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_PATH.exists(), f"Dataset not found at {RAW_PATH}"
print("Project root:", PROJECT_ROOT)
print("Running on Colab:", IN_COLAB)

Project root: /Users/dammamonnankulama/FYP/fyp-poc-v4
Running on Colab: False


In [2]:
import json
import hashlib
from datetime import datetime, timezone
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

raw = pd.read_csv(RAW_PATH)
df = raw.copy()
print(f"Raw shape: {raw.shape}")

Raw shape: (12316, 32)


## Step 1: Normalise text
Strip leading/trailing whitespace and collapse repeated internal spaces in every text column. For example, `"  Recreational areas"` and `"Recreational areas"` become one category.

In [3]:
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
before = df[cat_cols].nunique()

for c in cat_cols:
    df[c] = df[c].str.strip().str.replace(r"\s{2,}", " ", regex=True)

after = df[cat_cols].nunique()
changed = pd.DataFrame({"unique_before": before, "unique_after": after})
changed[changed.unique_before != changed.unique_after]

,unique_before,unique_after
Area_accident_occured,14,13


## Step 2: Fix corrupted and inconsistent labels
Each correction is listed explicitly so it can be audited and reused on any new data, for example Sri Lankan records later.

In [4]:
LABEL_FIXES = {
    "Type_of_vehicle": {               # '?' was used in place of '-'
        "Lorry (41?100Q)": "Lorry (41-100Q)",
        "Lorry (11?40Q)": "Lorry (11-40Q)",
        "Public (13?45 seats)": "Public (13-45 seats)",
    },
    "Area_accident_occured": {         # two labels merged together -> ambiguous
        "Rural village areasOffice areas": "Other",
    },
    "Road_allignment": {               # truncated label
        "Tangent road with mountainous terrain and": "Tangent road with mountainous terrain",
    },
    "Lanes_or_Medians": {"other": "Other"},
    "Driving_experience": {"unknown": "Unknown"},
    "Fitness_of_casuality": {"NormalNormal": "Normal"},
}

for col, mapping in LABEL_FIXES.items():
    n = df[col].isin(mapping.keys()).sum()
    df[col] = df[col].replace(mapping)
    print(f"{col:<24} fixed {n:>4} values")

# 'stationary' was damaged by a find/replace of 'na' -> 'Not a Pedestrian' in the source data
n = df["Pedestrian_movement"].str.contains("statioNot a Pedestrianry", regex=False).sum()
df["Pedestrian_movement"] = df["Pedestrian_movement"].str.replace("statioNot a Pedestrianry", "stationary", regex=False)
print(f"{'Pedestrian_movement':<24} fixed {n:>4} values")

Type_of_vehicle          fixed 3259 values
Area_accident_occured    fixed   20 values
Road_allignment          fixed  396 values
Lanes_or_Medians         fixed 1660 values
Driving_experience       fixed   33 values
Fitness_of_casuality     fixed   19 values
Pedestrian_movement      fixed  468 values


In [5]:
# Rename misspelled column names for readability (original names kept in metadata)
COLUMN_RENAMES = {
    "Area_accident_occured": "Area_accident_occurred",
    "Road_allignment": "Road_alignment",
    "Work_of_casuality": "Work_of_casualty",
    "Fitness_of_casuality": "Fitness_of_casualty",
}
df = df.rename(columns=COLUMN_RENAMES)
cat_cols = [COLUMN_RENAMES.get(c, c) for c in cat_cols]
list(COLUMN_RENAMES.values())

['Area_accident_occurred',
 'Road_alignment',
 'Work_of_casualty',
 'Fitness_of_casualty']

## Step 3: Unify missing values → `Unknown`
NaN, `na` and `Unknown` all mean *"not recorded"*. We merge them into **one explicit `Unknown` category** rather than imputing the most frequent value, for three reasons:
- Imputing with the mode would push even more rows into the already-dominant categories and invent information we don't have.
- "Not recorded" may itself carry signal (records may be less complete for some accident types). An explicit category lets the model learn that.
- Dropping rows with any missing value would remove a large share of the data, including scarce **fatal** cases.

In [6]:
MISSING_TOKENS = ["na", "NA", "unknown", "Unknown"]
missing_before = (df[cat_cols].isna() | df[cat_cols].isin(MISSING_TOKENS)).sum()

df[cat_cols] = df[cat_cols].replace(MISSING_TOKENS, np.nan).fillna("Unknown")

print("Remaining NaN values:", int(df.isna().sum().sum()))
missing_before[missing_before > 0].sort_values(ascending=False).to_frame("values_now_'Unknown'")

Remaining NaN values: 0


,values_now_'Unknown'
Service_year_of_vehicle,6811
Casualty_class,4443
Casualty_severity,4443
Age_band_of_casualty,4443
Sex_of_casualty,4443
Defect_of_vehicle,4427
Work_of_casualty,3217
Fitness_of_casualty,2635
Age_band_of_driver,1548
Types_of_Junction,1078


## Step 4: Feature engineering from time
EDA showed that the fatal rate rises sharply at night. `Time` is a raw string with 1,074 distinct values, so we turn it into features a model can use:

| New feature | Definition |
|---|---|
| `hour` | Hour of day, 0–23 (integer) |
| `time_of_day` | Night (00–05), Morning (06–11), Afternoon (12–16), Evening (17–20), Late evening (21–23) |
| `is_night` | 1 if hour ≥ 19 or hour < 6 (matches the darkness-related severity pattern) |
| `is_weekend` | 1 if Saturday or Sunday |

In [7]:
t = pd.to_datetime(df["Time"], format="%H:%M:%S", errors="coerce")
assert t.notna().all(), "Unparseable Time values found"

df["hour"] = t.dt.hour.astype(int)
df["time_of_day"] = pd.cut(df["hour"], bins=[-1, 5, 11, 16, 20, 23],
                           labels=["Night", "Morning", "Afternoon", "Evening", "Late evening"]).astype(str)
df["is_night"] = ((df["hour"] >= 19) | (df["hour"] < 6)).astype(int)
df["is_weekend"] = df["Day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

# Sanity check: does the engineered feature reproduce the EDA finding?
chk = pd.crosstab(df["time_of_day"], df["Accident_severity"], normalize="index").mul(100).round(2)
chk = chk.reindex(["Night", "Morning", "Afternoon", "Evening", "Late evening"])
chk["n"] = df["time_of_day"].value_counts()
chk

Accident_severity,Fatal injury,Serious Injury,Slight Injury,n
time_of_day,,,,
Night,2.52,17.63,79.85,675
Morning,0.77,13.20,86.03,3236
Afternoon,1.00,13.73,85.27,3897
Evening,1.37,14.07,84.55,3496
Late evening,2.87,16.80,80.34,1012


## Step 5: Encode the target
Severity is **ordinal**, so we store an integer code alongside the original label:
`Slight Injury → 0`, `Serious Injury → 1`, `Fatal injury → 2`.

In [8]:
TARGET = "Accident_severity"
TARGET_MAP = {"Slight Injury": 0, "Serious Injury": 1, "Fatal injury": 2}

df["severity_code"] = df[TARGET].map(TARGET_MAP)
assert df["severity_code"].notna().all()
df["severity_code"] = df["severity_code"].astype(int)
df.groupby(["severity_code", TARGET]).size().to_frame("count")

,,count
severity_code,Accident_severity,
0,Slight Injury,10415
1,Serious Injury,1743
2,Fatal injury,158


## Step 6: Feature selection for Model 1 (Accident Severity Predictor)
This follows the leakage review in notebook 01 §7 and Proposal Appendix E. Only **context available at the moment of a violation** is kept.

In [9]:
MODEL1_FEATURES = {
    "time":        ["hour", "time_of_day", "is_night", "Day_of_week", "is_weekend"],
    "environment": ["Light_conditions", "Weather_conditions", "Road_surface_conditions"],
    "road":        ["Road_surface_type", "Road_alignment", "Lanes_or_Medians",
                    "Types_of_Junction", "Area_accident_occurred"],
    "driver":      ["Age_band_of_driver", "Driving_experience"],
    "vehicle":     ["Type_of_vehicle"],
    "violation":   ["Cause_of_accident"],
}
DROPPED = {
    "post_accident_leakage": ["Number_of_casualties", "Number_of_vehicles_involved", "Type_of_collision",
                              "Casualty_class", "Sex_of_casualty", "Age_band_of_casualty", "Casualty_severity",
                              "Work_of_casualty", "Fitness_of_casualty", "Pedestrian_movement"],
    "sensitive_attribute":   ["Sex_of_driver", "Educational_level"],
    "low_quality":           ["Service_year_of_vehicle", "Defect_of_vehicle"],
    "not_relevant":          ["Vehicle_driver_relation", "Owner_of_vehicle", "Vehicle_movement"],
    "replaced_by_engineered":["Time"],
}

feature_cols = [c for group in MODEL1_FEATURES.values() for c in group]
numeric_features = ["hour", "is_night", "is_weekend"]
categorical_features = [c for c in feature_cols if c not in numeric_features]

# Every original column must be accounted for: either kept or explicitly dropped
all_dropped = [c for group in DROPPED.values() for c in group]
original_cols = [COLUMN_RENAMES.get(c, c) for c in raw.columns]
unaccounted = set(original_cols) - set(feature_cols) - set(all_dropped) - {TARGET}
assert not unaccounted, f"Unaccounted columns: {unaccounted}"

model1 = df[feature_cols + [TARGET, "severity_code"]].copy()
print(f"Model 1 dataset: {model1.shape[0]:,} rows × {len(feature_cols)} features (+ target)")
print(f"  categorical: {len(categorical_features)}   numeric: {len(numeric_features)}")
model1.head()

Model 1 dataset: 12,316 rows × 17 features (+ target)
  categorical: 14   numeric: 3


,hour,time_of_day,is_night,Day_of_week,is_weekend,Light_conditions,Weather_conditions,Road_surface_conditions,Road_surface_type,Road_alignment,Lanes_or_Medians,Types_of_Junction,Area_accident_occurred,Age_band_of_driver,Driving_experience,Type_of_vehicle,Cause_of_accident,Accident_severity,severity_code
0,17,Evening,0,Monday,0,Daylight,Normal,Dry,Asphalt roads,Tangent road with flat terrain,Unknown,No junction,Residential areas,18-30,1-2yr,Automobile,Moving Backward,Slight Injury,0
1,17,Evening,0,Monday,0,Daylight,Normal,Dry,Asphalt roads,Tangent road with flat terrain,Undivided Two way,No junction,Office areas,31-50,Above 10yr,Public (> 45 seats),Overtaking,Slight Injury,0
2,17,Evening,0,Monday,0,Daylight,Normal,Dry,Asphalt roads,Unknown,Other,No junction,Recreational areas,18-30,1-2yr,Lorry (41-100Q),Changing lane to the left,Serious Injury,1
3,1,Night,1,Sunday,1,Darkness - lights lit,Normal,Dry,Earth roads,Tangent road with mild grade and flat terrain,Other,Y Shape,Office areas,18-30,5-10yr,Public (> 45 seats),Changing lane to the right,Slight Injury,0
4,1,Night,1,Sunday,1,Darkness - lights lit,Normal,Dry,Asphalt roads,Tangent road with flat terrain,Other,Y Shape,Industrial areas,18-30,2-5yr,Unknown,Overtaking,Slight Injury,0


In [10]:
# Cardinality of each categorical feature (number of one-hot columns it will produce later)
card = model1[categorical_features].nunique().sort_values(ascending=False)
print("Total one-hot columns after encoding ≈", card.sum())
card.to_frame("n_categories")

Total one-hot columns after encoding ≈ 122


,n_categories
Cause_of_accident,20
Type_of_vehicle,18
Area_accident_occurred,12
Road_alignment,10
Weather_conditions,9
Types_of_Junction,8
Day_of_week,7
Lanes_or_Medians,7
Driving_experience,7
Road_surface_type,6


## Step 7: Validation checks

In [11]:
checks = {
    "row count unchanged":                len(df) == len(raw),
    "no NaN anywhere":                    df.isna().sum().sum() == 0,
    "no leading/trailing whitespace":     all((df[c].astype(str) == df[c].astype(str).str.strip()).all() for c in cat_cols),
    "no '?' corruption left":             not df[cat_cols].apply(lambda s: s.str.contains("?", regex=False)).any().any(),
    "no 'statioNot' corruption left":     not df["Pedestrian_movement"].str.contains("statioNot", regex=False).any(),
    "no lowercase 'unknown'/'other'":     not df[cat_cols].isin(["unknown", "other", "na"]).any().any(),
    "target distribution unchanged":      raw[TARGET].value_counts().equals(df[TARGET].value_counts()),
    "3 target classes":                   df["severity_code"].nunique() == 3,
    "no leakage columns in Model 1 set":  not set(DROPPED["post_accident_leakage"]) & set(model1.columns),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values()), "Validation failed"

PASS  row count unchanged
PASS  no NaN anywhere
PASS  no leading/trailing whitespace
PASS  no '?' corruption left
PASS  no 'statioNot' corruption left
PASS  no lowercase 'unknown'/'other'
PASS  target distribution unchanged
PASS  3 target classes
PASS  no leakage columns in Model 1 set


In [12]:
# Target distribution is preserved exactly (no rows lost or altered)
pd.DataFrame({
    "raw":       raw[TARGET].value_counts(),
    "processed": model1[TARGET].value_counts(),
    "percent":   (model1[TARGET].value_counts(normalize=True) * 100).round(2),
})

,raw,processed,percent
Accident_severity,,,
Slight Injury,10415,10415,84.56
Serious Injury,1743,1743,14.15
Fatal injury,158,158,1.28


## Step 8: Save processed data and metadata
| File | Contents | Used by |
|---|---|---|
| `rta_cleaned_full.csv` | All columns, cleaned, plus engineered features | Further EDA; later models (e.g. driver risk) |
| `rta_model1.csv` | Model 1 features + target only | Severity predictor training (notebook 03) |
| `preprocessing_metadata.json` | Feature lists, target mapping, label fixes, dropped columns and reasons, source-file hash | Reproducibility (NFR-4); reuse of the same pipeline on Sri Lankan data |

In [13]:
full_path   = PROCESSED_DIR / "rta_cleaned_full.csv"
model1_path = PROCESSED_DIR / "rta_model1.csv"
meta_path   = PROCESSED_DIR / "preprocessing_metadata.json"

df.to_csv(full_path, index=False)
model1.to_csv(model1_path, index=False)

metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source_file": RAW_PATH.name,
    "source_sha256": hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    "source_licence": "CC BY 4.0 (Addis Ababa Sub-City police RTA dataset, Kaggle)",
    "n_rows": int(len(model1)),
    "target": TARGET,
    "target_code_column": "severity_code",
    "target_mapping": TARGET_MAP,
    "class_counts": {k: int(v) for k, v in model1[TARGET].value_counts().items()},
    "model1_features": MODEL1_FEATURES,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "dropped_columns": DROPPED,
    "column_renames": COLUMN_RENAMES,
    "label_fixes": LABEL_FIXES,
    "missing_tokens_mapped_to_Unknown": MISSING_TOKENS + ["NaN"],
    "pandas_version": pd.__version__,
}
meta_path.write_text(json.dumps(metadata, indent=2))

for p in (full_path, model1_path, meta_path):
    print(f"saved  {p.relative_to(PROJECT_ROOT)}  ({p.stat().st_size / 1024:,.0f} KB)")

saved  data/processed/rta_cleaned_full.csv  (4,615 KB)
saved  data/processed/rta_model1.csv  (2,504 KB)
saved  data/processed/preprocessing_metadata.json  (3 KB)


In [14]:
# Round-trip check: reload the saved file and confirm it matches
reloaded = pd.read_csv(model1_path)
assert reloaded.shape == model1.shape
assert (reloaded["severity_code"].values == model1["severity_code"].values).all()
print("Reload OK:", reloaded.shape)

Reload OK: (12316, 19)


## Summary
- **12,316 rows kept** (none dropped), so all **158 fatal** and **1,743 serious** cases are preserved.
- Labels were normalised and corrected, and all missing markers were unified to `Unknown`.
- Four time features were engineered (`hour`, `time_of_day`, `is_night`, `is_weekend`).
- **17 leakage-free context features** were selected for Model 1, following Appendix E.
- Outputs are saved in `data/processed/` with a metadata file for reproducibility.

### Next: `03_baseline_models.ipynb`
1. Stratified train/test split (fixed seed, as in Proposal §3.7).
2. `ColumnTransformer` (one-hot encoding for categoricals) inside an sklearn `Pipeline`.
3. Re-run the POC baselines (RF, XGBoost, LR, MLP, KNN, majority) and report balanced accuracy, macro-F1 and per-class recall/precision.
4. Add imbalance handling **on the training split only**: class weights and SMOTENC (needs `imbalanced-learn`).